# Loading dataset

## Download from kaggle

!pip install kagglehub
!pip install pandas

In [ ]:
# import kagglehub

# # Download latest version
# archive_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

## convert to sqlite DB

In [2]:
import sqlite3
import pandas as pd
import os

# Set up paths
archive_path = r"D:\2026\MLOps\MLOps-Qafza-2026\Tasks\dataset\archive"
db_path = r"D:\2026\MLOps\MLOps-Qafza-2026\Tasks\dataset\olist.db"

# Connect to SQLite database
conn = sqlite3.connect(db_path)

# Get all CSV files in the archive folder
csv_files = [f for f in os.listdir(archive_path) if f.endswith('.csv')]

print("Loading CSV files into SQLite database...")
print("=" * 50)

# Loop through each CSV file
for csv_file in csv_files:
    # Remove '_dataset.csv' or '.csv' from filename to use as table name
    if '_dataset.csv' in csv_file:
        table_name = csv_file.replace('_dataset.csv', '')
    else:
        table_name = csv_file.replace('.csv', '')
    
    # Full path to the CSV file
    file_path = os.path.join(archive_path, csv_file)
    
    # Read CSV
    df = pd.read_csv(file_path)
    
    # Save to SQLite
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    
    print(f"Loaded: {csv_file} -> table: {table_name} ({len(df)} rows)")

# Close connection
conn.close()

print("=" * 50)
print("All files loaded successfully!")
print(f"Database saved to: {db_path}")

# Reconnect to verify and show tables
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("\nTables in database:")
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
    count = cursor.fetchone()[0]
    print(f"   • {table[0]}: {count:,} rows")

conn.close()

Loading CSV files into SQLite database...
Loaded: olist_customers_dataset.csv -> table: olist_customers (99441 rows)
Loaded: olist_geolocation_dataset.csv -> table: olist_geolocation (1000163 rows)
Loaded: olist_orders_dataset.csv -> table: olist_orders (99441 rows)
Loaded: olist_order_items_dataset.csv -> table: olist_order_items (112650 rows)
Loaded: olist_order_payments_dataset.csv -> table: olist_order_payments (103886 rows)
Loaded: olist_order_reviews_dataset.csv -> table: olist_order_reviews (99224 rows)
Loaded: olist_products_dataset.csv -> table: olist_products (32951 rows)
Loaded: olist_sellers_dataset.csv -> table: olist_sellers (3095 rows)
Loaded: product_category_name_translation.csv -> table: product_category_name_translation (71 rows)
All files loaded successfully!
Database saved to: D:\2026\MLOps\MLOps-Qafza-2026\Tasks\dataset\olist.db

Tables in database:
   • olist_customers: 99,441 rows
   • olist_geolocation: 1,000,163 rows
   • olist_orders: 99,441 rows
   • olist_o

# Understanding dataset

```mermaid
erDiagram
    OLIST_CUSTOMERS ||--o{ OLIST_ORDERS : "places"
    OLIST_ORDERS ||--o{ OLIST_ORDER_ITEMS : "contains"
    OLIST_ORDERS ||--o{ OLIST_ORDER_PAYMENTS : "has"
    OLIST_ORDERS ||--o{ OLIST_ORDER_REVIEWS : "receives"
    OLIST_ORDER_ITEMS }o--|| OLIST_PRODUCTS : "references"
    OLIST_ORDER_ITEMS }o--|| OLIST_SELLERS : "sold_by"
    OLIST_PRODUCTS }o--|| OLIST_CATEGORY : "belongs_to"
    OLIST_CUSTOMERS }o--|| OLIST_GEOLOCATION : "located_in"
    OLIST_SELLERS }o--|| OLIST_GEOLOCATION : "located_in"

    OLIST_CUSTOMERS {
        string customer_id PK
        string customer_unique_id
        string customer_zip_code_prefix
        string customer_city
        string customer_state
    }

    OLIST_ORDERS {
        string order_id PK
        string customer_id FK
        string order_status
        timestamp order_purchase_timestamp
        timestamp order_approved_at
        timestamp order_delivered_carrier_date
        timestamp order_delivered_customer_date
        timestamp order_estimated_delivery_date
    }

    OLIST_ORDER_ITEMS {
        string order_id FK
        int order_item_id
        string product_id FK
        string seller_id FK
        timestamp shipping_limit_date
        float price
        float freight_value
    }

    OLIST_ORDER_PAYMENTS {
        string order_id FK
        int payment_sequential
        string payment_type
        int payment_installments
        float payment_value
    }

    OLIST_ORDER_REVIEWS {
        string review_id PK
        string order_id FK
        int review_score
        string review_comment_title
        string review_comment_message
        timestamp review_creation_date
        timestamp review_answer_timestamp
    }

    OLIST_PRODUCTS {
        string product_id PK
        string product_category_name FK
        int product_name_length
        int product_description_length
        int product_photos_qty
        int product_weight_g
        int product_length_cm
        int product_height_cm
        int product_width_cm
    }

    OLIST_SELLERS {
        string seller_id PK
        string seller_zip_code_prefix
        string seller_city
        string seller_state
    }

    OLIST_CATEGORY {
        string product_category_name PK
        string product_category_name_english
    }

    OLIST_GEOLOCATION {
        string geolocation_zip_code_prefix
        float geolocation_lat
        float geolocation_lng
        string geolocation_city
        string geolocation_state
    }

# Olist E-Commerce Database: Table Relationships & Summary

## Relationships Explanation

### 1. **Core Relationship: Customers ↔ Orders**
- **Type:** One-to-Many (1:N)
- **Key:** `customer_id`
- **Explanation:** One customer can place multiple orders, but each order belongs to exactly one customer. This is the foundation of the customer- order relationship.

### 2. **Orders ↔ Order Items**
- **Type:** One-to-Many (1:N)
- **Key:** `order_id`
- **Explanation:** One order can contain multiple items (products), and each item record belongs to exactly one order. This is why we need to aggregate items when creating ML features per order.

### 3. **Order Items ↔ Products**
- **Type:** Many-to-One (N:1)
- **Key:** `product_id`
- **Explanation:** Multiple order items can reference the same product (same product sold in different orders), but each item is for one specific product.

### 4. **Order Items ↔ Sellers**
- **Type:** Many-to-One (N:1)
- **Key:** `seller_id`
- **Explanation:** A seller can sell many items across different orders, but each item is sold by exactly one seller.

### 5. **Orders ↔ Payments**
- **Type:** One-to-Many (1:N)
- **Key:** `order_id`
- **Explanation:** An order can have multiple payment installments (e.g., credit card split into 3 payments). This is another table that needs aggregation per order.

### 6. **Orders ↔ Reviews**
- **Type:** One-to-One (1:1) typically
- **Key:** `order_id`
- **Explanation:** Usually one review per order, though some orders may not have reviews. Important for ML: reviews happen AFTER delivery, so they can cause data leakage if used for prediction.

### 7. **Products ↔ Category**
- **Type:** Many-to-One (N:1)
- **Key:** `product_category_name`
- **Explanation:** Many products belong to the same category (e.g., many products in "electronics" category). The translation table converts Portuguese category names to English.

### 8. **Geolocation ↔ Customers/Sellers**
- **Type:** Reference table
- **Key:** `zip_code_prefix`
- **Explanation:** Provides geographical coordinates for locations. Can be joined with customers and sellers through zip codes.

---

## Database Summary

The Olist dataset represents a **real Brazilian e-commerce marketplace** with 9 interconnected tables:

| Table | Purpose | Rows (approx) | Primary Key |
|-------|---------|---------------|-------------|
| **olist_orders** | Main order transactions | 100,000+ | order_id |
| **olist_customers** | Customer profiles | 100,000+ | customer_id |
| **olist_order_items** | Products in each order | 110,000+ | (composite) |
| **olist_products** | Product catalog | 30,000+ | product_id |
| **olist_sellers** | Seller information | 3,000+ | seller_id |
| **olist_order_payments** | Payment transactions | 100,000+ | (composite) |
| **olist_order_reviews** | Customer feedback | 100,000+ | review_id |
| **olist_geolocation** | Location coordinates | 1,000,000+ | zip_code |
| **olist_category_translation** | Category names | 70+ | category_name |



# Perform Simple Joins on tables

In [7]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect(db_path)

print("PERFORMING JOINS BETWEEN TABLES")
print("=" * 60)

# JOIN 1: Orders with Customers
print("\nJOIN: orders + customers")
print("-" * 40)
query = """
SELECT 
    o.order_id,
    o.order_status,
    c.customer_city,
    c.customer_state
FROM olist_orders o
JOIN olist_customers c ON o.customer_id = c.customer_id
LIMIT 5;
"""
df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

# JOIN 2: Orders with Items and Products
print("\n2JOIN: orders + order_items + products")
print("-" * 40)
query = """
SELECT 
    o.order_id,
    oi.product_id,
    p.product_category_name,
    oi.price,
    oi.freight_value
FROM olist_orders o
JOIN olist_order_items oi ON o.order_id = oi.order_id
JOIN olist_products p ON oi.product_id = p.product_id
LIMIT 5;
"""
df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

# JOIN 3: Orders with Items and Sellers
print("\nJOIN: order_items + sellers")
print("-" * 40)
query = """
SELECT 
    oi.order_id,
    oi.seller_id,
    s.seller_city,
    s.seller_state,
    oi.price
FROM olist_order_items oi
JOIN olist_sellers s ON oi.seller_id = s.seller_id
LIMIT 5;
"""
df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

# JOIN 4: Orders with Payments
print("\nJOIN: orders + order_payments")
print("-" * 40)
query = """
SELECT 
    o.order_id,
    op.payment_type,
    op.payment_value,
    op.payment_installments
FROM olist_orders o
JOIN olist_order_payments op ON o.order_id = op.order_id
LIMIT 5;
"""
df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

# JOIN 5: Orders with Reviews
print("\nJOIN: orders + order_reviews")
print("-" * 40)
query = """
SELECT 
    o.order_id,
    orv.review_score,
    orv.review_comment_message
FROM olist_orders o
JOIN olist_order_reviews orv ON o.order_id = orv.order_id
LIMIT 5;
"""
df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

# JOIN 6: Complete order details (4 tables)
print("\nJOIN: orders + customers + order_items + products")
print("-" * 40)
query = """
SELECT 
    o.order_id,
    c.customer_city,
    p.product_category_name,
    oi.price,
    oi.freight_value
FROM olist_orders o
JOIN olist_customers c ON o.customer_id = c.customer_id
JOIN olist_order_items oi ON o.order_id = oi.order_id
JOIN olist_products p ON oi.product_id = p.product_id
LIMIT 5;
"""
df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

# Check relationship: one order can have multiple items
print("\nRelationship check: Orders with multiple items")
print("-" * 40)
query = """
SELECT 
    o.order_id,
    COUNT(oi.order_id) as item_count
FROM olist_orders o
JOIN olist_order_items oi ON o.order_id = oi.order_id
GROUP BY o.order_id
ORDER BY item_count DESC
LIMIT 5;
"""
df = pd.read_sql_query(query, conn)
print(df.to_string(index=False))

conn.close()

print("\n" + "=" * 60)
print("ALL JOINS COMPLETED SUCCESSFULLY!")

PERFORMING JOINS BETWEEN TABLES

JOIN: orders + customers
----------------------------------------
                        order_id order_status           customer_city customer_state
e481f51cbdc54678b7cc49136f2d6af7    delivered               sao paulo             SP
53cdb2fc8bc7dce0b6741e2150273451    delivered               barreiras             BA
47770eb9100c2d0c44946d9cf07ec65d    delivered              vianopolis             GO
949d5b44dbf5de918fe9c16f97b45f8a    delivered sao goncalo do amarante             RN
ad21c59c0840e6cb83a9ceb5573f8159    delivered             santo andre             SP

2JOIN: orders + order_items + products
----------------------------------------
                        order_id                       product_id product_category_name  price  freight_value
e481f51cbdc54678b7cc49136f2d6af7 87285b34884572647811a353c7ac498a utilidades_domesticas  29.99           8.72
53cdb2fc8bc7dce0b6741e2150273451 595fac2a385ac33a80bd5114aec74eb8            perfumaria 11